In [0]:
%sql
USE CATALOG adb_dream_analytics_prod;
CREATE SCHEMA IF NOT EXISTS gold MANAGED LOCATION 'abfss://gold@adlsgen2dreamprod.dfs.core.windows.net/dream_app/';
USE SCHEMA gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.dim_locations (
    location_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    location_id STRING,
    city STRING,
    state STRING,
    country STRING,
    latitude DECIMAL(11,8),
    longitude DECIMAL(11,8),
    last_updated_timestamp TIMESTAMP,
    scd1_hash STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    _processed_at TIMESTAMP
) USING DELTA;
    
CREATE OR REPLACE TEMPORARY VIEW v_locations_source AS
SELECT
    location_id,
    city,
    state,
    country,
    latitude,
    longitude,
    try_cast(last_updated_timestamp AS TIMESTAMP) AS last_updated_timestamp,
    md5(concat_ws('|', 
        upper(trim(city)), 
        upper(trim(state)), 
        upper(trim(country)), 
        latitude, 
        longitude
    )) AS source_hash,
    current_timestamp() AS _processed_at
FROM adb_dream_analytics_prod.silver.locations;
    
MERGE INTO dim_locations AS target
USING v_locations_source AS source
ON target.location_id = source.location_id
WHEN MATCHED AND target.scd1_hash <> source.source_hash THEN
    UPDATE SET
        target.city = source.city,
        target.state = source.state,
        target.country = source.country,
        target.latitude = source.latitude,
        target.longitude = source.longitude,
        target.scd1_hash = source.source_hash,
        target._processed_at = source._processed_at,
        target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
    INSERT (location_id, city, state, country, latitude, longitude, last_updated_timestamp, scd1_hash, _processed_at, created_at, updated_at)
    VALUES (source.location_id, source.city, source.state, source.country, source.latitude, source.longitude, source.last_updated_timestamp, source.source_hash, source._processed_at, current_timestamp(), current_timestamp());